# Instalando as Bibliotecas Necessárias



In [ ]:
# Installing the numpy, netcdf4, boto3 and gdal libraries
!pip install -q cartopy boto3 gdal salem rasterio pyproj geopandas descartes

# download do arquivo "utilities_goes16e19.py"
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/utilities_goes16e19.py

# download da paleta de cores para o canal do infravermelho
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/ir.cpt

# Monta drive
#from google.colab import drive
#drive.mount('/content/drive')

# Caminho do diretório
#dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/4_SATELITE/codigos_imagem_satelite_REFERENCIA'

# Diretório de Saída
import os
#dir_output = f'{dir}/output/SAT_10'
#os.makedirs(dir_output, exist_ok=True)

# Baixando as Imagens

In [ ]:
#========================================================================================================================#
#                                          IMPORTAÇÃO DAS BIBLIOTECAS
#========================================================================================================================#
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
import cartopy, cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from datetime import datetime
from utilities_goes16e19 import download_CMI, remap, loadCPT
import numpy as np
import os
import pandas as pd
import folium
import numpy as np
import matplotlib.pyplot as plt
import branca.colormap as bcm
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import box

#========================================================================================================================#
#                                        CRIA DIRETÓRIO DE ENTRADA E SAÍDA
#========================================================================================================================#
input = "/content/input"; os.makedirs(input, exist_ok=True)
#output = dir_output

#========================================================================================================================#
#                                        DEFINE OS LIMITES DA IMAGEM
#========================================================================================================================#
# canal
band = '13'

# área desejada da imagem
lonmin, lonmax, latmin, latmax = -100., -25.24, -56.00, 12.52
#lonmin, lonmax, latmin, latmax = -139.4 , -11.07, -57.6, 57.52

# # coloca os limites da área numa lista
extent = [lonmin, latmin, lonmax, latmax]

#========================================================================================================================#
#                                               DOWNLOAD DO ARQUIVO
#========================================================================================================================#
# datas
datas = ['202603211000', '202603211010', '202603211030']
datas = ['202603271740', '202603271750', '202603271800']

# Lista para armazenar as imagens e informações
imagens = []

# Loop das imagens
for i, data in enumerate(datas):

    #--------------------------------------------------------------------------#
    #                          DATA E HORÁRIO
    #--------------------------------------------------------------------------#
    # define o satélite: GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(data, '%Y%m%d%H%M')
    goes_number = '16' if imagem_atual < start_g19 else '19'

    # extrai o ano, mês, dia, hor e min
    ano = datetime.strptime(data, '%Y%m%d%H%M').strftime('%Y')
    mes= datetime.strptime(data, '%Y%m%d%H%M').strftime('%m')
    dia = datetime.strptime(data, '%Y%m%d%H%M').strftime('%d')
    hor = datetime.strptime(data, '%Y%m%d%H%M').strftime('%H')
    min = datetime.strptime(data, '%Y%m%d%H%M').strftime('%M')

    print('#=====================================================================================================#')
    print(f'                          PROCESSANDO A IMAGEM = {ano}-{mes}-{dia} {hor}{min} UTC'                       )
    print('#=====================================================================================================#')

    # download do arquivo
    file_name = download_CMI(data, band, goes_number, input)

    # caminho do arquivo que foi baixado
    path = f'{input}/{file_name}.nc'

    #--------------------------------------------------------------------------#
    #                    REPROJETA OS DADOS DO ABI
    #--------------------------------------------------------------------------#
    # chama a função que faz a reprojeção (file, variable, extent, resolution)
    grid = remap(path, 'CMI', extent, 2)

    # remove os arquivos desnecessários
    !rm {input}/{file_name}.nc
    !rm {input}/{file_name}.nc.aux.xml

    #--------------------------------------------------------------------------#
    #                    LEITURA ARQUIVO NETCDF
    #--------------------------------------------------------------------------#
    ds = xr.open_dataset(f'{input}/{file_name}_ret.nc', mask_and_scale=True).sel(lon=slice(lonmin, lonmax), lat=slice(latmax, latmin))

    # extrair dados e converter de Kelvin para Celsius
    data_k = ds['Band1'].values/10.
    data_c = data_k - 273.15  # Conversão direta

    # extrai as latitude,longitudes e
    lat = ds['lat'].values
    lon = ds['lon'].values

    # Armazena os dados para uso posterior
    imagens.append({
        'data': data,
        'data_c': data_c,
        'lat': lat,
        'lon': lon,
        'hora': f'{hor}:{min}'
    })

# Plota Mapa Interativo

In [ ]:
# ========================================================================================================================
#                                     PLOTA MAPA INTERATIVO COM 3 CAMADAS
# ========================================================================================================================

# ------------------------------------------------------------------------------------------------------------------------
# 1. CONFIGURAÇÕES INICIAIS
# ------------------------------------------------------------------------------------------------------------------------
# Limites de temperatura e níveis do colormap
vmin, vmax = -103.0, 84
levels = np.arange(vmin, vmax, 1.0)

# Carrega o colormap do arquivo CPT (padrão GOES)
cpt = loadCPT('ir.cpt')
colormap = cm.colors.LinearSegmentedColormap('cpt', cpt)
norm = mcolors.BoundaryNorm(levels, colormap.N)

# Limites geográficos da imagem (lonmin, lonmax, latmin, latmax já definidos anteriormente)
bounds = [[latmin, lonmin], [latmax, lonmax]]
bbox = box(lonmin, latmin, lonmax, latmax)

# ------------------------------------------------------------------------------------------------------------------------
# 2. CRIA MAPA BASE
# ------------------------------------------------------------------------------------------------------------------------
m = folium.Map(location=[(latmin + latmax)/2, (lonmin + lonmax)/2],
               zoom_start=4, tiles='OpenStreetMap', control_scale=True)

# ------------------------------------------------------------------------------------------------------------------------
# 3. FUNÇÕES AUXILIARES PARA PLOTAR CONTORNOS
# ------------------------------------------------------------------------------------------------------------------------
def add_contour(geometry, color='white', weight=1.0, opacity=0.7):

    """Adiciona contorno de uma geometria ao mapa Folium"""

    coords_list = []

    # Extrai coordenadas de polígonos simples ou múltiplos
    if geometry.geom_type == 'Polygon':
        coords_list = [list(geometry.exterior.coords)]
    elif geometry.geom_type == 'MultiPolygon':
        coords_list = [list(poly.exterior.coords) for poly in geometry.geoms]

    # Adiciona cada polígono ao mapa
    for coords in coords_list:
        coords_swap = [(lat, lon) for lon, lat in coords]  # Inverte (lon,lat) -> (lat,lon)
        folium.PolyLine(coords_swap, color=color, weight=weight, opacity=opacity).add_to(m)

def load_and_filter_geodata(source, bbox, filter_col=None, filter_val=None):

    """Carrega dados geoespaciais e filtra pelo bounding box"""
    try:
        if isinstance(source, str):  # Se for URL ou caminho
            gdf = gpd.read_file(source)
        else:  # Se for shapefile do Natural Earth
            gdf = gpd.read_file(source)

        # Aplica filtro adicional se necessário
        if filter_col and filter_val:
            gdf = gdf[gdf[filter_col].isin(filter_val)] if isinstance(filter_val, list) else gdf[gdf[filter_col] == filter_val]

        # Filtra pela área da imagem
        return gdf[gdf.geometry.intersects(bbox)]
    except Exception as e:
        print(f"Erro ao carregar dados: {e}")
        return None

# ------------------------------------------------------------------------------------------------------------------------
# 4. ADICIONA CONTORNOS DOS PAÍSES
# ------------------------------------------------------------------------------------------------------------------------
print("Carregando contornos dos países...")
world = gpd.read_file(shpreader.natural_earth(resolution='110m', category='cultural', name='admin_0_countries'))

# Filtra países da América do Sul e do Norte/Central
south_america = world[world['CONTINENT'] == 'South America']
north_central = world[world['CONTINENT'].isin(['North America'])]
north_central = north_central[north_central.geometry.bounds['minx'] <= lonmax]  # Filtra por longitude
countries = pd.concat([south_america, north_central])

# Plota países dentro da área
for _, country in countries.iterrows():
    if country.geometry.intersects(bbox):
        add_contour(country.geometry, color='white', weight=1.5, opacity=0.8)

# ------------------------------------------------------------------------------------------------------------------------
# 5. ADICIONA CONTORNOS DOS ESTADOS BRASILEIROS
# ------------------------------------------------------------------------------------------------------------------------
print("Carregando contornos dos estados brasileiros...")
try:
    url_states = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
    brazil_states = gpd.read_file(url_states)
    states_in_area = brazil_states[brazil_states.geometry.intersects(bbox)]

    for _, state in states_in_area.iterrows():
        add_contour(state.geometry, color='white', weight=1.0, opacity=0.7)

except Exception as e:
    print(f"Erro nos estados brasileiros: {e}")
    # Fallback: usa apenas o contorno do Brasil
    brazil = world[world['SOVEREIGNT'] == 'Brazil']
    if not brazil.empty and brazil.geometry.iloc[0].intersects(bbox):
        add_contour(brazil.geometry.iloc[0], color='white', weight=1.5, opacity=0.8)

# ------------------------------------------------------------------------------------------------------------------------
# 6. ADICIONA CONTORNOS DE ESTADOS/PROVÍNCIAS (EUA/CANADÁ)
# ------------------------------------------------------------------------------------------------------------------------
print("Carregando contornos de estados/províncias...")
try:
    states_shp = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_1_states_provinces')
    states = gpd.read_file(states_shp)
    states_in_area = states[states.geometry.intersects(bbox)]

    for _, state in states_in_area.iterrows():
        add_contour(state.geometry, color='white', weight=0.8, opacity=0.5)

except Exception as e:
    print(f"Erro nos estados/províncias: {e}")

# ------------------------------------------------------------------------------------------------------------------------
# 7. ADICIONA AS CAMADAS DE IMAGEM (3 DATAS)
# ------------------------------------------------------------------------------------------------------------------------
print("Adicionando camadas de imagens de satélite...")

for i, img in enumerate(imagens):

    # Prepara os dados para visualização
    data_discretized = np.digitize(img['data_c'], levels) - 1
    data_discretized = np.ma.masked_where(data_discretized < 0, data_discretized)

    # Gera a imagem RGBA e salva temporariamente
    rgba_image = colormap(data_discretized / colormap.N)
    temp_file = f'goes_ch13_celsius_{i}.png'
    plt.imsave(temp_file, rgba_image)

    # Formata a data para legenda
    data_obj = datetime.strptime(img['data'], '%Y%m%d%H%M')
    data_formatada = data_obj.strftime('%Y-%m-%d %H:%M UTC')

    # Adiciona overlay da imagem
    folium.raster_layers.ImageOverlay(image=temp_file, bounds=bounds, opacity=0.7,
                                      name=data_formatada, interactive=True).add_to(m)

# ------------------------------------------------------------------------------------------------------------------------
# 8. CRIA A BARRA DE CORES
# ------------------------------------------------------------------------------------------------------------------------
print("Criando barra de cores...")

# Extrai cores em formato hexadecimal
colors_hex = []
for i, level in enumerate(levels):

    pos = (level - vmin) / (vmax - vmin) if i < len(levels)-1 else 1.0
    colors_hex.append(mcolors.to_hex(colormap(pos)))

# Remove duplicatas e cria a barra
colors_hex = list(dict.fromkeys(colors_hex))
step_cmap = bcm.LinearColormap(colors=colors_hex, vmin=vmin, vmax=vmax,
                               caption='Temperatura de Brilho (°C)')
step_cmap.add_to(m)

# ------------------------------------------------------------------------------------------------------------------------
# 9. ESTILIZA A BARRA DE CORES COM CSS
# ------------------------------------------------------------------------------------------------------------------------
css_style = """
<style>
.legend .tick text { font-size: 14px !important; font-weight: bold !important; fill: #000000 !important; }
.legend .caption text { font-size: 18px !important; font-weight: bold !important; fill: #000000 !important; }
.legend .tick line { stroke-width: 2px !important; stroke: #000000 !important; }
.legend .colorbar { stroke-width: 2px !important; stroke: #000000 !important; }
</style>
"""
m.get_root().html.add_child(folium.Element(css_style))

# ------------------------------------------------------------------------------------------------------------------------
# 10. ADICIONA TÍTULO E CONTROLE DE CAMADAS
# ------------------------------------------------------------------------------------------------------------------------
titulo_html = f'''
<h3 align="center" style="font-size:18px; font-weight:bold;">
<b>GOES-{goes_number} Canal 13 (IR)</b><br>
<span style="font-size:14px;">Limites: Lon: {lonmin:.1f}° a {lonmax:.1f}° | Lat: {latmin:.1f}° a {latmax:.1f}°</span>
</h3>
'''

m.get_root().html.add_child(folium.Element(titulo_html))
folium.LayerControl(collapsed=False).add_to(m)

# ------------------------------------------------------------------------------------------------------------------------
# 11. SALVA E EXIBE O MAPA
# ------------------------------------------------------------------------------------------------------------------------
m.save('mapa_goes_celsius_3camadas.html')
display(m)

print("\n" + "="*60)
print("✓ MAPA GERADO COM SUCESSO!")
print(f"  • Camadas: {len(imagens)} imagens de satélite")
print(f"  • Limites: Lon: {lonmin} a {lonmax} | Lat: {latmin} a {latmax}")
print(f"  • Arquivo: mapa_goes_celsius_3camadas.html")
print("="*60)